# 08 — Metric aggregation across folds

This notebook confirms the cross-fold aggregation used by `CrossValidationReport`. We hand-construct per-fold metric dictionaries with known values, then call the report's real `_mean_std`, `_scalar_keys`, and `_aggregate_table` methods and check the mean and sample standard deviation against values computed independently with numpy.

Modules exercised: `pipelines.cross_validation_pipeline.cv_report.CrossValidationReport`, `pipelines.benchmark_pipeline.results.TrialRecord`, `pipelines.cross_validation_pipeline.folds.FoldPlanner`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except Exception:
    torch = None

SEED = 0
RNG  = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.titlesize" : 11,
    "axes.labelsize" : 10,
    "image.cmap"     : "viridis",
})

print("Bootstrap complete. Repository root on sys.path:", Path("../..").resolve())


## Hand-construct per-fold metric records

Each fold gets a `TrialRecord` carrying a small metric dictionary. The values are chosen so the mean and standard deviation are easy to anticipate.

In [ ]:
import tempfile

from configuration.cross_validation_config import CrossValidationConfig
from pipelines.cross_validation_pipeline.folds import FoldPlanner
from pipelines.cross_validation_pipeline.cv_report import CrossValidationReport
from pipelines.benchmark_pipeline.results import TrialRecord
from tools.logger import Logger

N_FOLDS = 5
config                     = CrossValidationConfig()
config.folds.n_folds       = N_FOLDS
config.folds.azimuth_start = 0
config.folds.azimuth_end   = 500
planner = FoldPlanner(config, range_start=0, range_end=100)

metric_values = {
    "curve_rmse_gt"    : [0.50, 0.52, 0.48, 0.55, 0.45],
    "overall_r2_gt"    : [0.90, 0.88, 0.91, 0.85, 0.92],
    "pixel_r2_gt_mean" : [0.80, 0.82, 0.79, 0.83, 0.81],
}

def make_records():
    records = []
    for fold_index in range(N_FOLDS):
        record = TrialRecord(name=f"fold_{fold_index}", run_dir=Path("/synthetic"))
        record.metrics = {key: values[fold_index] for key, values in metric_values.items()}
        records.append(record)
    return records

records_by_split = {"val": make_records(), "test": make_records()}
base_records     = [TrialRecord(name=f"fold_{i}", run_dir=Path("/synthetic")) for i in range(N_FOLDS)]
print("per-fold metrics ready for", N_FOLDS, "folds")

## Instantiate the real report object

In [ ]:
tmp_dir = Path(tempfile.mkdtemp())
logger  = Logger(log_dir=str(tmp_dir), name="cv_aggregation_demo")

report = CrossValidationReport(
    base_records     = base_records,
    records_by_split = records_by_split,
    planner          = planner,
    out_dir          = tmp_dir / "reports",
    model_name       = "resunet",
    embed_images     = False,
    logger           = logger,
)

print("scalar keys discovered:", report._scalar_keys(records_by_split["test"]))

## Compare the report's mean and std against numpy

`_mean_std` uses the sample standard deviation (denominator `n - 1`), which corresponds to numpy's `ddof=1`.

In [ ]:
print(f"{'metric':18s} {'report_mean':>12s} {'numpy_mean':>12s} {'report_std':>12s} {'numpy_std':>12s} {'match':>7s}")
for key, values in metric_values.items():
    report_mean, report_std = report._mean_std(values)
    numpy_mean = float(np.mean(values))
    numpy_std  = float(np.std(values, ddof=1))
    match      = np.isclose(report_mean, numpy_mean) and np.isclose(report_std, numpy_std)
    print(f"{key:18s} {report_mean:12.6f} {numpy_mean:12.6f} {report_std:12.6f} {numpy_std:12.6f} {str(match):>7s}")

## Render the aggregate table and visualise mean with std error bars

We render the report's real markdown aggregate table, then plot the per-metric mean across folds with the sample standard deviation as an error bar.

In [ ]:
keys  = report._scalar_keys(records_by_split["test"])
table = report._aggregate_table(keys, records_by_split["test"])
print("\n".join(table))

In [ ]:
means = [report._mean_std(metric_values[k])[0] for k in metric_values]
stds  = [report._mean_std(metric_values[k])[1] for k in metric_values]

x = np.arange(len(metric_values))
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(x, means, yerr=stds, capsize=6, color="#4878a8", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(list(metric_values.keys()), rotation=15, ha="right")
ax.set_ylabel("value")
ax.set_title("Cross-fold metric means with sample std error bars")
plt.tight_layout()
plt.show()

logger.close()

## Expected visual outcome

Every metric reports `match = True`, confirming the report's `_mean_std` agrees with numpy's sample mean and standard deviation. The rendered markdown table lists each metric with its mean, std, and per-fold columns, and the bar chart shows the three metric means with std error bars.